In [1]:
%pip install datasets sentence_transformers lazypredict
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Note: you may need to restart the kernel to use updated packages.


In [1]:
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
import numpy as np
import datasets
import pandas as pd
from sentence_transformers import SentenceTransformer
import lazypredict
import torch

df = pd.read_csv('dataset_clinica20252.csv', sep='|')
df = df[df.ds_fatos.apply(lambda x: isinstance(x, str) and len(x) > 0)]
df.head()

,cd_causa,cd_atendimento,ds_Acao_Judicial,ds_fatos,ds_Pedidos,ds_Qualificacao
0,CIB0500064,0825789-84.2025.8.18.0140,90 - ACAO DE REPARACAO DE DANOS,"DOS FATOS A parte Autora, pessoa idosa e hipos...","DOS PEDIDOS Ante ao exposto, requer: a) Sejam ...",DOUTO JUÍZO DE DIREITO DA ___ VARA CÍVEL DA CO...
1,CIB0505587,1004697-72.2025.8.26.0066,90 - ACAO DE REPARACAO DE DANOS,"DOS FATOS 5. A parte autora é pessoa idosa, hi...",DOS PEDIDOS E REQUERIMENTOS 33. Diante do expo...,(17) 99779-9177 / EXCELENTÍSSIMO SENHOR DOUTOR...
2,CIB0508201,0800423-07.2025.8.15.0761,90 - ACAO DE REPARACAO DE DANOS,DOS FATOS 1. SITUAÇÃO DE VULNERABILIDADE DO CO...,"DOS PEDIDOS E REQUERIMENTOS Ex Positis, requer...",AO COLENDO JUÍZO DA VARA ÚNICA DA COMARCA DE G...
3,CIB0514647,1004875-69.2025.8.26.0438,90 - ACAO DE REPARACAO DE DANOS,"DOS FATOS De plano, necessário esclarecer que ...","DOS PEDIDOS Diante do exposto, requer-se a Vos...","Avenida Eduardo de Castilho, n.o 315, Centro, ..."
4,CIB0500604,0010630-50.2025.8.27.2706,90 - ACAO DE REPARACAO DE DANOS,DOS FATOS A parte Requerente é correntista usu...,DOS PEDIDOS 1. QUE SEJA DEFERIDA A GRATUIDADE ...,AO DOUTO JUÍZO DA VARA CÍVEL DA COMARCA DE ARA...


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_experimental.text_splitter import SemanticChunker

docs = [texto for texto in df.ds_fatos if isinstance(texto, str)]
model_name = "rufimelo/bert-large-portuguese-cased-sts"
# model_kwargs = {'device': 'cpu'}
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': False}
embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs,
    )

embedding_vector = embeddings.embed_documents(docs)
print(f"Embedding Vector: {embedding_vector[0]}")

Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.


Embedding Vector: [-0.34810060262680054, -0.8226074576377869, 0.350344717502594, -0.5188853144645691, 0.3259665369987488, 0.1912507861852646, 0.2786983251571655, -0.4937233328819275, -0.17999151349067688, 0.043286845088005066, -0.5072917342185974, -0.9959406852722168, -0.33212316036224365, -0.2087884247303009, 0.0513310432434082, 0.3459709584712982, -0.07529139518737793, -0.060564614832401276, -0.19605879485607147, 0.5959060192108154, 0.1025269478559494, -0.04418246075510979, 0.09470786154270172, 0.48720335960388184, 0.32662126421928406, 0.18862998485565186, 0.0623217336833477, 0.17982113361358643, -0.37635791301727295, 0.13515573740005493, -0.42435216903686523, 0.5047026872634888, -0.9168495535850525, -0.7580335140228271, -0.19898344576358795, -0.045325472950935364, -0.33249998092651367, 0.02688795141875744, -0.16353444755077362, -0.11122811585664749, -0.488877534866333, 0.0027717496268451214, -0.23469671607017517, 0.13130474090576172, 0.8376961946487427, 0.20897287130355835, -0.04720

In [3]:
import re

classificacao = list()
for _,row in df.iterrows():
    check = re.search(r'cr[eé]dito\sconsignado', row['ds_fatos'], re.IGNORECASE|re.S)
    if check:
        classificacao.append('consignado')
    else:
        classificacao.append('outros')
df['classificacao'] = classificacao

In [13]:
le = LabelEncoder()
df['classificacao'] = le.fit_transform(df['classificacao'])

# split into train, validate, test
train, validate, test = \
              np.split(df.sample(frac=1, random_state=42),
                       [int(.2*len(df)), int(.2*len(df))])

# reset indices
train = train.reset_index()[['ds_fatos','classificacao']][:100]
test = test.reset_index()[['ds_fatos','classificacao']][:100]
validate = validate.reset_index()[['ds_fatos','classificacao']]

# dataframes to datadict
tds = Dataset.from_pandas(train)
testds = Dataset.from_pandas(test)
vds = Dataset.from_pandas(validate)
dataset = datasets.DatasetDict()
dataset['train'] = tds
dataset['test'] = testds
dataset['validation'] = vds

In [15]:
model_name = 'stjiris/bert-large-portuguese-cased-legal-tsdae-gpl-nli-sts-MetaKD-v0'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SentenceTransformer(model_name, device=device)
train_embeddings = model.encode(dataset['train']['ds_fatos'], show_progress_bar=True)
test_embeddings = model.encode(dataset['test']['ds_fatos'], show_progress_bar=True)
validate_embeddings = model.encode(dataset['validation']['ds_fatos'], show_progress_bar=True)
train_embeddings = np.array(train_embeddings)
test_embeddings = np.array(test_embeddings)
validate_embeddings = np.array(validate_embeddings)

Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches: 0it [00:00, ?it/s]

In [16]:
# %pip install lazypredict
from lazypredict.Supervised import LazyClassifier
import numpy as np
import random

train_labels = np.array(dataset['train']['classificacao'])
test_labels = np.array(dataset['test']['classificacao'])
validate_labels = np.array(dataset['validation']['classificacao'])

# sample_size = int(0.33 * len(train_embeddings))
# random_indices = random.sample(range(len(train_embeddings)), sample_size)
# sample_train_embeddings = train_embeddings[random_indices]
# sample_train_labels = train_labels[random_indices]

In [7]:
import lazypredict
from lazypredict.Supervised import LazyClassifier
# classifiers = ['PassiveAggressiveClassifier','LogisticRegression',
#  'LinearDiscriminantAnalysis','RidgeClassifierCV','RidgeClassifier',
#  'Perceptron','SGDClassifier','RandomForestClassifier','ExtraTreesClassifier']
# no_classifiers = ['MLPClassifier','LogisticRegressionCV','GradientBoostingClassifier','HistGradientBoostingClassifier','AdaBoostClassifier']
# list_classifiers = []
# for item in lazypredict.Supervised.CLASSIFIERS:
#     if item[0] not in no_classifiers:
#         list_classifiers.append(item)
# lazypredict.Supervised.CLASSIFIERS = list_classifiers

In [8]:
# clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
clf = LazyClassifier()
models, predictions = clf.fit(train_embeddings, test_embeddings,  train_labels , test_labels)
# models, predictions = clf.fit(sample_train_embeddings, test_embeddings,  sample_train_labels , test_labels)
print(models)

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 11620, number of negative: 259
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.053478 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 261120
[LightGBM] [Info] Number of data points in the train set: 11879, number of used features: 1024
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.978197 -> initscore=3.803655
[LightGBM] [Info] Start training from score 3.803655
                               Accuracy  Balanced Accuracy  ROC AUC  F1 Score  \
Model                                                                           
LogisticRegression                 0.98               0.73     0.73      0.98   
PassiveAggressiveClassifier        0.98               0.72     0.72      0.98   
LinearSVC                          0.97               0.72     0.72      0.97   
NearestCentroid                    0.82               0.71     0.71      0.89   
LinearDiscriminantAn